# Week 3 — Sentiment Analysis & Urgency Scoring
## AI-Driven Citizen Grievance & Sentiment Analysis System

**Objective:** Assign an urgency/sentiment score to every citizen complaint so civic officials can instantly triage critical issues from routine ones — without reading every submission manually.

**What we build this week:**
- Rule-based sentiment baselines: **VADER** and **TextBlob**
- A labelled urgency dataset with 4 classes: `Low`, `Medium`, `High`, `Critical`
- A supervised **Logistic Regression** urgency classifier on TF-IDF features
- Fine-tuning **DistilBERT** for contextual sentiment understanding
- A unified **Dual-Output prediction function** — department + urgency in one call
- Visualizations: urgency distribution, sentiment heatmap by department, model comparison
- Save all models for Week 4 FastAPI deployment

---
**Roadmap:**
1. Install & Import Libraries
2. Load Data & Week 2 Models
3. Rule-Based Baselines — VADER & TextBlob
4. Urgency Label Engineering
5. EDA — Sentiment & Urgency Distributions
6. Supervised Urgency Classifier (TF-IDF + Logistic Regression)
7. DistilBERT Fine-Tuning
8. Model Evaluation & Comparison
9. Dual-Output Prediction Pipeline Demo
10. Save All Models for Week 4

---
## Step 1: Install & Import Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn joblib tqdm
!pip install vaderSentiment textblob
!pip install transformers datasets torch accelerate
!python -m textblob.download_corpora

In [ ]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm import tqdm
warnings.filterwarnings('ignore')

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, pipeline as hf_pipeline
)
from datasets import Dataset

random.seed(42); np.random.seed(42); torch.manual_seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'All libraries loaded. Device: {DEVICE.upper()}')

---
## Step 2: Load Data & Week 2 Models

In [ ]:
# ── Load processed dataset ───────────────────────────────────────
if os.path.exists('processed_grievances.csv'):
    df = pd.read_csv('processed_grievances.csv')
    print(f'Loaded processed_grievances.csv — Shape: {df.shape}')
else:
    print('processed_grievances.csv not found — regenerating synthetic dataset...')
    import spacy, nltk
    from nltk.corpus import stopwords
    nltk.download('stopwords', quiet=True)
    !python -m spacy download en_core_web_sm -q
    nlp_spacy = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    STOPWORDS = set(stopwords.words('english'))
    templates = {
        'Water Supply':      ['Water supply cut off for {n} days.','Tap water has foul smell, undrinkable.','No running water since {n} days. Residents struggling.','Water leakage from pipeline wasting thousands of liters.','Contaminated water causing illness in neighborhood.','Water tanker not arrived for {n} days despite requests.'],
        'Electricity':       ['Power outage for {n} hours, no communication from utility.','Voltage fluctuations damaging home appliances.','Street lights in sector {n} not working for weeks.','Electric pole dangerously tilted, could collapse.','Transformer blown. No power since {n} days.','Live wire exposed near playground, risk to children.'],
        'Roads & Transport': ['Massive potholes on main road damaged several vehicles.','Road not repaired despite {n} complaints filed.','No footpath. Pedestrians forced to walk on road.','Bus route {n} cancelled without announcement.','Traffic signal at main junction not functioning.','No speed breaker near school zone. Accidents frequent.'],
        'Sanitation':        ['Garbage not collected for {n} days. Stench unbearable.','Open drain next to residential area is health hazard.','Waste dumped near park attracting rats and animals.','Sewage overflow on main street. Urgent action required.','Community dustbin overflowing and spreading disease.','Dead animals not removed from street for {n} days.'],
        'Public Safety':     ['Street lights not working. Area unsafe at night.','Frequent theft in sector {n}. Need police patrol.','Stray dogs attacking children near school.','Illegal construction blocking emergency access road.','Drug peddling happening openly near the park.','Speeding vehicles are daily danger on inner road.'],
    }
    rows = []
    for dept, phrases in templates.items():
        for _ in range(250):
            phrase = random.choice(phrases).format(n=random.randint(2, 15))
            rows.append({'narrative': phrase, 'product': dept})
    df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
    def clean_text(t):
        t = re.sub(r'http\S+', '', str(t).lower())
        t = re.sub(r'[^a-z\s]', '', t)
        return re.sub(r'\s+', ' ', t).strip()
    df['cleaned_text'] = df['narrative'].apply(clean_text)
    processed = []
    for doc in tqdm(nlp_spacy.pipe(df['cleaned_text'], batch_size=500), total=len(df)):
        tokens = [t.lemma_ for t in doc if t.text not in STOPWORDS and len(t.text) > 2]
        processed.append(' '.join(tokens))
    df['processed_text'] = processed
    df.to_csv('processed_grievances.csv', index=False)
    print(f'Synthetic dataset generated — Shape: {df.shape}')

# ── Load Week 2 department classifier ───────────────────────────────
if os.path.exists('best_department_classifier.pkl') and os.path.exists('label_encoder.pkl'):
    dept_pipeline = joblib.load('best_department_classifier.pkl')
    dept_le       = joblib.load('label_encoder.pkl')
    print('Week 2 department classifier loaded.')
else:
    print('Week 2 models not found — run Week 2 notebook first.')
    dept_pipeline = None; dept_le = None

print(f'Columns: {list(df.columns)} | Shape: {df.shape}')
df.head(3)

---
## Step 3: Rule-Based Sentiment Baselines

Before training any model we apply two fast rule-based sentiment tools as interpretable baselines and feature signals.

### 3a — VADER (Valence Aware Dictionary and sEntiment Reasoner)
Specifically designed for short, social-media-style text. Special handling for CAPITALIZATION (`URGENT` scores higher), punctuation emphasis (`!!!` amplifies negativity), and negation (`not good` handled correctly). Returns `neg`, `neu`, `pos`, `compound` scores (compound range: −1 to +1).

### 3b — TextBlob
Pattern-library based. Returns **polarity** (−1 to +1) and **subjectivity** (0=factual, 1=opinion). Civic complaints are typically high-subjectivity and negative — subjectivity distinguishes emotional venting from factual reporting.

In [ ]:
vader = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    s = vader.polarity_scores(str(text))
    return pd.Series({'vader_compound': s['compound'], 'vader_neg': s['neg'],
                      'vader_neu': s['neu'], 'vader_pos': s['pos']})

def get_textblob_scores(text):
    b = TextBlob(str(text))
    return pd.Series({'tb_polarity': b.sentiment.polarity,
                      'tb_subjectivity': b.sentiment.subjectivity})

tqdm.pandas(desc='VADER')
df = pd.concat([df, df['narrative'].progress_apply(get_vader_scores)], axis=1)
tqdm.pandas(desc='TextBlob')
df = pd.concat([df, df['narrative'].progress_apply(get_textblob_scores)], axis=1)

print('Sentiment scores added.')
print(f'Mean VADER compound : {df["vader_compound"].mean():.4f}')
print(f'Mean TB polarity    : {df["tb_polarity"].mean():.4f}')
print(f'Mean TB subjectivity: {df["tb_subjectivity"].mean():.4f}')
df[['narrative','vader_compound','vader_neg','tb_polarity','tb_subjectivity']].head(4)

---
## Step 4: Urgency Label Engineering

Urgency labels are assigned via a **hybrid strategy** — keyword rules + VADER compound score.

| Level | Meaning | Example |
|---|---|---|
| **Critical** | Immediate danger to life or health | Live wire exposed near playground |
| **High** | Significant disruption, needs fast response | No water for 7 days |
| **Medium** | Noticeable issue, moderate urgency | Road pothole causing vehicle damage |
| **Low** | Minor inconvenience | Garbage bin slightly overflowing |

In [ ]:
CRITICAL_KEYWORDS = [
    'dangerous','danger','collapse','emergency','urgent','life','death','die',
    'dying','fatal','attack','attacking','injured','injury','accident',
    'electrocute','fire','flood','live wire','exposed wire','child','children',
    'school','hospital','disease','illness','contaminate','contaminated',
    'sewage overflow','poison','toxic','gas leak'
]
HIGH_KEYWORDS = [
    'no water','no power','power cut','outage','cut off','broken',
    'not working','failed','failure','damage','damaged','repeated',
    'harassment','theft','stolen','crime','illegal','overflow',
    'stench','unbearable','hazard','health hazard','rats','no response'
]
MEDIUM_KEYWORDS = [
    'pothole','delay','slow','inconvenient','not cleaned','smell',
    'noise','complaint','issue','problem','request','pending',
    'fluctuation','low pressure','no footpath','no signal'
]

def assign_urgency(row):
    text_l   = str(row['narrative']).lower()
    compound = row['vader_compound']
    if any(kw in text_l for kw in CRITICAL_KEYWORDS): return 'Critical'
    if any(kw in text_l for kw in HIGH_KEYWORDS) or compound <= -0.6: return 'High'
    if any(kw in text_l for kw in MEDIUM_KEYWORDS) or compound <= -0.2: return 'Medium'
    return 'Low'

df['urgency_label'] = df.apply(assign_urgency, axis=1)
print('Urgency labels assigned:')
print(df['urgency_label'].value_counts())

---
## Step 5: EDA — Sentiment & Urgency Distributions

In [ ]:
urgency_order  = ['Critical','High','Medium','Low']
urgency_colors = {'Critical':'#e74c3c','High':'#e67e22','Medium':'#f1c40f','Low':'#2ecc71'}

# ── Plot 1: Urgency distribution bar + pie ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['urgency_label'].value_counts().reindex(urgency_order)
bars = axes[0].bar(counts.index, counts.values,
                   color=[urgency_colors[u] for u in counts.index], edgecolor='white')
for bar in bars:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 str(int(bar.get_height())), ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Urgency Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Complaints')
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[urgency_colors[u] for u in urgency_order], startangle=140,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Urgency — Proportional Breakdown', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('urgency_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: urgency_distribution.png')

In [ ]:
# ── Plot 2: VADER compound distribution by urgency ───────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for label in urgency_order:
    subset = df[df['urgency_label']==label]['vader_compound']
    ax.hist(subset, bins=30, alpha=0.6, label=label, color=urgency_colors[label], edgecolor='white')
ax.axvline(0, color='black', linestyle='--', linewidth=1.2, label='Neutral (0)')
ax.set_xlabel('VADER Compound Score', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('VADER Score Distribution by Urgency Level', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('vader_by_urgency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: vader_by_urgency.png')

In [ ]:
# ── Plot 3: Sentiment heatmap by department ──────────────────────────────────
dept_sentiment = df.groupby('product')[['vader_compound','vader_neg','tb_polarity','tb_subjectivity']].mean().round(3)
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(dept_sentiment, annot=True, fmt='.3f', cmap='RdYlGn', linewidths=0.5,
            ax=ax, cbar_kws={'shrink':0.8})
ax.set_title('Average Sentiment Scores by Department', fontsize=13, fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('sentiment_heatmap_by_dept.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sentiment_heatmap_by_dept.png')

In [ ]:
# ── Plot 4: Urgency breakdown per department (stacked bar) ───────────────────
pivot = df.groupby(['product','urgency_label']).size().unstack(fill_value=0).reindex(columns=urgency_order, fill_value=0)
fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(pivot))
for u in urgency_order:
    vals = pivot[u].values
    ax.bar(pivot.index, vals, bottom=bottom, label=u, color=urgency_colors[u], edgecolor='white', linewidth=0.8)
    bottom += vals
ax.set_xlabel('Department', fontsize=12)
ax.set_ylabel('Number of Complaints', fontsize=12)
ax.set_title('Urgency Level Breakdown per Department', fontsize=13, fontweight='bold')
ax.legend(title='Urgency', bbox_to_anchor=(1.01,1), loc='upper left')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('urgency_by_department.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: urgency_by_department.png')

In [ ]:
# ── Plot 5: VADER vs TextBlob scatter by urgency ─────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for label in urgency_order:
    sub = df[df['urgency_label']==label]
    ax.scatter(sub['vader_compound'], sub['tb_polarity'], alpha=0.4,
               label=label, color=urgency_colors[label], s=18)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel('VADER Compound Score', fontsize=12)
ax.set_ylabel('TextBlob Polarity', fontsize=12)
ax.set_title('VADER vs TextBlob — Sentiment Agreement by Urgency', fontsize=13, fontweight='bold')
ax.legend(title='Urgency', markerscale=2)
plt.tight_layout()
plt.savefig('vader_vs_textblob_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: vader_vs_textblob_scatter.png')

---
## Step 6: Supervised Urgency Classifier (TF-IDF + Logistic Regression)

Rule-based tools score words in isolation — they have no memory of what a civic complaint looks like. A supervised classifier **learns** complaint-specific patterns from labelled examples.

We use `class_weight='balanced'` to handle the fact that `Critical` complaints are naturally less frequent than `Low` — without it the model ignores minority classes.

In [ ]:
urgency_le = LabelEncoder()
urgency_le.fit(urgency_order)
df['urgency_encoded'] = urgency_le.transform(df['urgency_label'])
print('Urgency encoding:')
for i, cls in enumerate(urgency_le.classes_): print(f'  {i} -> {cls}')

In [ ]:
X_text = df['processed_text']
y_urg  = df['urgency_encoded']

X_tr, X_te, y_tr, y_te = train_test_split(X_text, y_urg, test_size=0.2, random_state=42, stratify=y_urg)
print(f'Train: {len(X_tr)} | Test: {len(X_te)}')

In [ ]:
urgency_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=10000, sublinear_tf=True, min_df=2)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced',
                                  solver='lbfgs', multi_class='multinomial', random_state=42))
])

urgency_pipeline.fit(X_tr, y_tr)
y_pred_urg = urgency_pipeline.predict(X_te)
urg_acc    = accuracy_score(y_te, y_pred_urg)
urg_macro  = f1_score(y_te, y_pred_urg, average='macro')

print(f'Urgency Classifier — Accuracy: {urg_acc:.4f} | Macro F1: {urg_macro:.4f}')
print(classification_report(y_te, y_pred_urg, target_names=urgency_le.classes_))

In [ ]:
# 5-Fold Cross-Validation
from sklearn.feature_extraction.text import TfidfVectorizer as TFIDF2
tfidf_cv = TFIDF2(ngram_range=(1,2), max_features=10000, sublinear_tf=True, min_df=2)
X_all_tfidf = tfidf_cv.fit_transform(X_text)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced',
                       solver='lbfgs', multi_class='multinomial', random_state=42),
    X_all_tfidf, y_urg, cv=skf, scoring='f1_macro', n_jobs=-1
)
print(f'5-Fold CV Macro F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_te, y_pred_urg)
ConfusionMatrixDisplay(cm, display_labels=urgency_le.classes_).plot(
    ax=ax, cmap='Oranges', colorbar=True, xticks_rotation=15)
ax.set_title('Confusion Matrix — TF-IDF + LR Urgency Classifier', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('urgency_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: urgency_confusion_matrix.png')

---
## Step 7: DistilBERT Fine-Tuning for Contextual Sentiment

### Why BERT over TF-IDF for sentiment?
TF-IDF treats words independently. Consider:
- `"The road is not dangerous"` — TF-IDF sees `dangerous` → incorrectly scores Critical
- `"The road is not dangerous"` — DistilBERT understands `not dangerous` → correctly scores Low

**DistilBERT** is a distilled version of BERT — 60% smaller, 97% of the performance. **Fine-tuning** adapts its general language understanding to civic complaint vocabulary and urgency patterns specifically.

> Note: GPU (Colab T4/A100) recommended. On CPU, reduce `num_train_epochs` to 1 for a quick test.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
NUM_LABELS = len(urgency_order)
MAX_LENGTH = 128

df_bert = df[['narrative','urgency_encoded']].rename(columns={'narrative':'text','urgency_encoded':'label'})
train_df, test_df = train_test_split(df_bert, test_size=0.2, random_state=42, stratify=df_bert['label'])
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['label'])

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

train_tok = train_dataset.map(tokenize_fn, batched=True)
val_tok   = val_dataset.map(tokenize_fn, batched=True)
test_tok  = test_dataset.map(tokenize_fn, batched=True)

keep = ['input_ids','attention_mask','label']
for ds in [train_tok, val_tok, test_tok]:
    ds.set_format('torch', columns=keep)

print('Tokenization complete.')

In [ ]:
id2label = {i: urgency_le.classes_[i] for i in range(NUM_LABELS)}
label2id = {v: k for k, v in id2label.items()}

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
).to(DEVICE)

trainable = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f'DistilBERT loaded. Trainable parameters: {trainable:,}')

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'macro_f1': f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir                  = './distilbert_urgency',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    learning_rate               = 2e-5,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    logging_steps               = 50,
    fp16                        = (DEVICE == 'cuda'),
    report_to                   = 'none',
)

trainer = Trainer(
    model=bert_model, args=training_args,
    train_dataset=train_tok, eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)
print(f'Ready to fine-tune on {DEVICE.upper()}. Epochs: {training_args.num_train_epochs}')

In [ ]:
train_result = trainer.train()
print(f'Fine-tuning complete! Training loss: {train_result.training_loss:.4f}')

In [ ]:
bert_preds_raw = trainer.predict(test_tok)
bert_preds     = np.argmax(bert_preds_raw.predictions, axis=-1)
bert_labels    = bert_preds_raw.label_ids
bert_acc   = accuracy_score(bert_labels, bert_preds)
bert_macro = f1_score(bert_labels, bert_preds, average='macro')

print(f'DistilBERT — Accuracy: {bert_acc:.4f} | Macro F1: {bert_macro:.4f}')
print(classification_report(bert_labels, bert_preds, target_names=urgency_le.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(bert_labels, bert_preds),
                       display_labels=urgency_le.classes_).plot(
    ax=ax, cmap='Purples', colorbar=True, xticks_rotation=15)
ax.set_title('Confusion Matrix — DistilBERT Fine-Tuned', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('bert_urgency_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: bert_urgency_confusion_matrix.png')

---
## Step 8: Model Evaluation & Comparison

In [ ]:
def vader_classify(compound):
    if compound <= -0.6: return urgency_le.transform(['High'])[0]
    if compound <= -0.2: return urgency_le.transform(['Medium'])[0]
    if compound > 0.0:   return urgency_le.transform(['Low'])[0]
    return urgency_le.transform(['Medium'])[0]

test_idx    = test_df.index
vader_preds = df.loc[test_idx, 'vader_compound'].apply(vader_classify).values
y_te_arr    = df.loc[test_idx, 'urgency_encoded'].values

comparison = pd.DataFrame([
    {'Model':'VADER (rule-based)',            'Accuracy':accuracy_score(y_te_arr,vader_preds), 'Macro F1':f1_score(y_te_arr,vader_preds,average='macro')},
    {'Model':'TF-IDF + Logistic Regression',  'Accuracy':urg_acc,   'Macro F1':urg_macro},
    {'Model':'DistilBERT Fine-Tuned',          'Accuracy':bert_acc,  'Macro F1':bert_macro},
])
print('Urgency Model Comparison:')
print(comparison.round(4).to_string(index=False))

In [ ]:
x = np.arange(len(comparison)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x-w/2, comparison['Accuracy'], w, label='Accuracy', color='#4e9af1', edgecolor='white')
b2 = ax.bar(x+w/2, comparison['Macro F1'], w, label='Macro F1', color='#f1c84e', edgecolor='white')
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'], rotation=10, ha='right')
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Urgency Classifier — Model Comparison', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('urgency_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: urgency_model_comparison.png')

---
## Step 9: Dual-Output Prediction Pipeline Demo

The full system now produces **two outputs** from one raw complaint:
1. **Department** — where to route it (Week 2 SVM model)
2. **Urgency** — how quickly to act on it (Week 3 DistilBERT model)

This is the core interface the civic dashboard will consume in Week 4.

In [ ]:
bert_clf = hf_pipeline(
    'text-classification', model=bert_model, tokenizer=tokenizer,
    device=0 if DEVICE=='cuda' else -1, return_all_scores=True
)

def preprocess_single(text):
    text = re.sub(r'http\S+','', str(text).lower())
    text = re.sub(r'[^a-z\s]','', text)
    return re.sub(r'\s+',' ', text).strip()

def analyze_complaint(raw_text, use_bert=True):
    cleaned = preprocess_single(raw_text)
    dept    = dept_le.inverse_transform(dept_pipeline.predict([cleaned]))[0] if dept_pipeline else 'N/A'
    if use_bert:
        res    = bert_clf(raw_text[:512])[0]
        best   = max(res, key=lambda x: x['score'])
        lid    = int(best['label'].split('_')[-1])
        urgency, conf = urgency_le.inverse_transform([lid])[0], round(best['score'], 3)
    else:
        enc    = urgency_pipeline.predict([cleaned])[0]
        urgency = urgency_le.inverse_transform([enc])[0]
        conf    = round(float(urgency_pipeline.predict_proba([cleaned])[0].max()), 3)
    compound = vader.polarity_scores(raw_text)['compound']
    return {'department': dept, 'urgency': urgency,
            'confidence': conf, 'vader_score': round(compound, 3)}

EMOJI = {'Critical':'[CRITICAL]','High':'[HIGH]','Medium':'[MEDIUM]','Low':'[LOW]'}
demo_complaints = [
    'A live electrical wire has fallen near the primary school. Children are in DANGER!!!',
    'No water supply for the last 8 days. We filed 5 complaints but nothing was done.',
    'Garbage bins near the market have not been collected this week.',
    'The traffic signal at main crossing is not working. Causing accidents daily.',
    'Sewage overflow near block C is spreading disease in the colony.',
    'Street lights were slightly dimmer than usual last night.',
]

print('DUAL-OUTPUT GRIEVANCE ANALYSIS DEMO')
print('='*65)
for c in demo_complaints:
    r = analyze_complaint(c, use_bert=True)
    print(f'\nComplaint  : {c[:80]}')
    print(f'Department : {r["department"]}')
    print(f'Urgency    : {EMOJI.get(r["urgency"],"")} {r["urgency"]}  (conf: {r["confidence"]})')
    print(f'VADER Score: {r["vader_score"]}')

---
## Step 10: Save All Models for Week 4

In [ ]:
joblib.dump(urgency_pipeline, 'urgency_classifier.pkl')
joblib.dump(urgency_le,       'urgency_label_encoder.pkl')

os.makedirs('distilbert_urgency_final', exist_ok=True)
bert_model.save_pretrained('distilbert_urgency_final')
tokenizer.save_pretrained('distilbert_urgency_final')

comparison.to_csv('week3_model_comparison.csv', index=False)
df[['narrative','processed_text','product','urgency_label',
    'vader_compound','tb_polarity','tb_subjectivity']].to_csv('week3_labelled_grievances.csv', index=False)

print('All Week 3 artifacts saved:')
print('  urgency_classifier.pkl         -- TF-IDF + LR urgency pipeline')
print('  urgency_label_encoder.pkl      -- Urgency label decoder')
print('  distilbert_urgency_final/      -- Fine-tuned DistilBERT + tokenizer')
print('  week3_model_comparison.csv     -- Performance comparison')
print('  week3_labelled_grievances.csv  -- Full labelled dataset')

---
## Summary — Week 3 Deliverables

| Deliverable | Status |
|---|---|
| VADER rule-based sentiment scoring | Done |
| TextBlob polarity & subjectivity | Done |
| Urgency label engineering (4-class hybrid) | Done |
| Urgency distribution & pie chart | Done |
| VADER score distribution by urgency | Done |
| Sentiment heatmap by department | Done |
| Urgency stacked bar by department | Done |
| VADER vs TextBlob scatter plot | Done |
| TF-IDF + Logistic Regression urgency classifier | Done |
| 5-Fold Stratified Cross-Validation | Done |
| DistilBERT fine-tuning (contextual sentiment) | Done |
| Confusion matrices for both models | Done |
| Model comparison bar chart | Done |
| Dual-output prediction demo | Done |
| All artifacts saved for Week 4 | Done |

**Next — Week 4:** Wrap both models in a **FastAPI** REST API with `POST /analyze` and `POST /batch_analyze` endpoints, then build a **Streamlit dashboard** for civic officials showing a live complaint queue sorted by urgency.